# 01 — Occupancy data audit

This notebook loads the three UCI Occupancy Detection files and performs a simple, time-aware first exploration. It intentionally does **not** train a model.

Goals:

- confirm that all source files load consistently;
- inspect shapes, types, missing values, duplicates, and ranges;
- preserve the supplied train/test identities;
- examine occupancy balance and sensor distributions;
- visualize temporal behavior and occupancy transitions;
- identify questions to carry into feature engineering and modeling.

## 1. Imports and display settings

In [ ]:
# Import filesystem, data-analysis, and interactive-visualization tools.
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

# Make tabular output easier to read inside the notebook.
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
pd.options.plotting.backend = "plotly"
# Apply a consistent visual theme and class colors to every Plotly chart.
PLOTLY_TEMPLATE = "plotly_white"
STATE_COLORS = {"Unoccupied": "#4C78A8", "Occupied": "#F58518"}

RANDOM_SEED = 42

## 2. Locate and load the source files

The path logic works whether Jupyter starts in the repository root or in `notebooks/`. The first CSV field has a blank header and is an exported row index; it is retained as `source_row_id` for auditing but will not be used as a model feature.

In [ ]:
# Find the repository root so paths work when Jupyter starts from either
# the project directory or the notebooks directory.
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root containing pyproject.toml")


# Build all data paths from the discovered root instead of the current directory.
PROJECT_ROOT = find_project_root()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

# Keep the source-provided split names attached to their physical files.
SOURCE_FILES = {
    "train": "datatraining.txt",
    "test_1": "datatest.txt",
    "test_2": "datatest2.txt",
}

# The source header omits the first field even though every data row starts
# with a row ID. Supplying all eight names prevents pandas from guessing that
# the first field should become the DataFrame index.
SOURCE_COLUMNS = [
    "source_row_id",
    "date",
    "Temperature",
    "Humidity",
    "Light",
    "CO2",
    "HumidityRatio",
    "Occupancy",
]

PROJECT_ROOT, RAW_DATA_DIR

In [ ]:
# Load one source CSV with an explicit schema and label its supplied split.
def load_source_file(path: Path, split_name: str) -> pd.DataFrame:
    frame = pd.read_csv(
        path,
        header=0,
        names=SOURCE_COLUMNS,
        parse_dates=["date"],
    )

    # Check the alignment immediately so a future source-format change fails
    # visibly instead of silently shifting values into the wrong columns.
    assert frame.columns.tolist() == SOURCE_COLUMNS
    assert frame["source_row_id"].notna().all()
    assert pd.api.types.is_datetime64_any_dtype(frame["date"])
    # This label lets us combine data for EDA without losing evaluation boundaries.
    frame["source_split"] = split_name
    return frame


# Load all files into a dictionary so each supplied period remains accessible.
frames = {
    split: load_source_file(RAW_DATA_DIR / filename, split)
    for split, filename in SOURCE_FILES.items()
}

# Preview each source independently before combining anything.
for split, frame in frames.items():
    print(f"{split:>6}: {frame.shape[0]:,} rows × {frame.shape[1]} columns")
    display(frame.head(3))

## 3. Schema and quality checks

In [ ]:
frames.items()

In [ ]:
# Define the data contract that every supplied file is expected to satisfy.
EXPECTED_COLUMNS = {
    "source_row_id",
    "date",
    "Temperature",
    "Humidity",
    "Light",
    "CO2",
    "HumidityRatio",
    "Occupancy",
    "source_split",
}

# Collect file-level quality checks into one compact audit table.
schema_summary = []
# Fail early if a schema, target value, or timestamp order is unexpected.
for split, frame in frames.items():
    schema_summary.append(
        {
            "split": split,
            "rows": len(frame),
            "columns": frame.shape[1],
            "schema_matches": set(frame.columns) == EXPECTED_COLUMNS,
            "missing_cells": int(frame.isna().sum().sum()),
            "duplicate_rows": int(frame.duplicated().sum()),
            "duplicate_timestamps": int(frame["date"].duplicated().sum()),
            "start": frame["date"].min(),
            "end": frame["date"].max(),
            "is_time_sorted": frame["date"].is_monotonic_increasing,
        }
    )

schema_summary = pd.DataFrame(schema_summary).set_index("split")
display(schema_summary)

for split, frame in frames.items():
    assert set(frame.columns) == EXPECTED_COLUMNS, f"Unexpected schema in {split}"
    assert set(frame["Occupancy"].dropna().unique()).issubset({0, 1})
    assert frame["date"].is_monotonic_increasing, f"Timestamps not sorted in {split}"

In [ ]:
# Combine the periods only for descriptive analysis and sort them by time.
# The source_split column is deliberately retained for later filtering.
data = (
    pd.concat(frames.values(), ignore_index=True)
    .sort_values("date")
    .reset_index(drop=True)
)

# Confirm the combined shape, inferred types, missingness, and earliest rows.
print(f"Combined shape: {data.shape}")
display(data.dtypes.rename("dtype").to_frame())
display(data.isna().sum().rename("missing").to_frame())
display(data.head())

The combined frame is useful for visualization, but `source_split` must remain intact. Model development should use `train`; the two test periods remain held out.

In [ ]:
# List genuine sensor inputs once so identifiers and the target stay excluded.
SENSOR_COLUMNS = [
    "Temperature",
    "Humidity",
    "Light",
    "CO2",
    "HumidityRatio",
]

# Review central tendency and spread for sensors and the binary target.
display(data[SENSOR_COLUMNS + ["Occupancy"]].describe().T)

# Extreme values and cardinality can reveal parsing problems or near-constant sensors.
range_summary = pd.DataFrame(
    {
        "minimum": data[SENSOR_COLUMNS].min(),
        "maximum": data[SENSOR_COLUMNS].max(),
        "unique_values": data[SENSOR_COLUMNS].nunique(),
    }
)
display(range_summary)

## 4. Occupancy balance

Inspect each supplied period separately. Similar overall class proportions do not guarantee similar sensor distributions or transition behavior.

In [ ]:
# Compare both absolute class counts and within-split occupancy rates.
class_counts = pd.crosstab(data["source_split"], data["Occupancy"])
class_rates = pd.crosstab(
    data["source_split"], data["Occupancy"], normalize="index"
)

display(class_counts)
display(class_rates.rename(columns={0: "unoccupied_rate", 1: "occupied_rate"}))

# Reshape the rate table into the long format expected by Plotly traces.
balance_plot = (
    class_rates.rename(columns={0: "Unoccupied", 1: "Occupied"})
    .rename_axis("source_split")
    .reset_index()
    .melt(id_vars="source_split", var_name="State", value_name="Share")
)

display(balance_plot)
# Build one stacked bar trace per occupancy state for interactive comparison.
fig = go.Figure()
for state in ["Unoccupied", "Occupied"]:
    subset = balance_plot.loc[balance_plot["State"] == state]
    fig.add_bar(
        x=subset["source_split"],
        y=subset["Share"],
        name=state,
        marker_color=STATE_COLORS[state],
        hovertemplate="%{x}<br>%{y:.1%}<extra>" + state + "</extra>",
    )
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    barmode="stack",
    title="Occupancy balance by supplied split",
    xaxis_title="Split",
    yaxis_title="Share",
)
fig.update_yaxes(tickformat=".0%")
fig.show()

## 5. Sensor distributions by occupancy

In [ ]:
# Reshape all sensor columns into a common sensor/value representation.
long_data = data.melt(
    id_vars=["Occupancy", "source_split"],
    value_vars=SENSOR_COLUMNS,
    var_name="sensor",
    value_name="value",
)

# Human-readable labels make chart legends clearer than numeric 0/1 values.
long_data["Occupancy label"] = long_data["Occupancy"].map(
    {0: "Unoccupied", 1: "Occupied"}
)

display(long_data)

# Give every sensor its own axis because their units and ranges differ greatly.
SENSOR_AXIS_LABELS = {
    "Temperature": "Temperature (°C)",
    "Humidity": "Relative humidity (%)",
    "Light": "Light (lux)",
    "CO2": "CO2 (ppm)",
    "HumidityRatio": "Humidity ratio (kg water/kg air)",
}

fig = make_subplots(rows=3, cols=2, subplot_titles=SENSOR_COLUMNS)
for index, sensor in enumerate(SENSOR_COLUMNS):
    row, col = divmod(index, 2)
    for state in ["Unoccupied", "Occupied"]:
        values = long_data.loc[
            (long_data["sensor"] == sensor)
            & (long_data["Occupancy label"] == state),
            "value",
        ]
        fig.add_trace(
            go.Histogram(
                x=values,
                name=state,
                histnorm="probability density",
                opacity=0.55,
                nbinsx=50,
                marker_color=STATE_COLORS[state],
                legendgroup=state,
                showlegend=index == 0,
            ),
            row=row + 1,
            col=col + 1,
        )
# Label each panel with the correct sensor unit and density scale.
for index, sensor in enumerate(SENSOR_COLUMNS):
    row, col = divmod(index, 2)
    fig.update_xaxes(
        title_text=SENSOR_AXIS_LABELS[sensor],
        row=row + 1,
        col=col + 1,
    )
    fig.update_yaxes(
        title_text="Probability density",
        row=row + 1,
        col=col + 1,
    )

fig.update_layout(
    template=PLOTLY_TEMPLATE,
    barmode="overlay",
    title="Sensor distributions by occupancy state",
    height=850,
)
fig.show()

In [ ]:
# Quantify the visual differences between occupied and unoccupied distributions.
group_summary = data.groupby("Occupancy")[SENSOR_COLUMNS].agg(["mean", "median", "std"])
display(group_summary)

## 6. Correlations and possible proxy features

`HumidityRatio` is derived from temperature and humidity, so strong correlation is expected. A very strong relationship between `Light` and `Occupancy` should motivate explicit no-light experiments.

In [ ]:
# Calculate linear associations among sensors and with the occupancy target.
correlations = data[SENSOR_COLUMNS + ["Occupancy"]].corr()

# Display the full matrix as an annotated interactive heatmap.
fig = go.Figure(
    go.Heatmap(
        z=correlations.to_numpy(),
        x=correlations.columns,
        y=correlations.index,
        colorscale="RdBu",
        reversescale=True,
        zmin=-1,
        zmax=1,
        text=np.round(correlations.to_numpy(), 2),
        texttemplate="%{text:.2f}",
        hovertemplate="%{y} vs %{x}<br>r=%{z:.3f}<extra></extra>",
    )
)
fig.update_layout(template=PLOTLY_TEMPLATE, title="Pearson correlation matrix", height=650)
fig.show()

# Rank target correlations by magnitude to highlight likely proxy features.
display(
    correlations["Occupancy"]
    .drop("Occupancy")
    .sort_values(key=abs, ascending=False)
    .rename("correlation_with_occupancy")
    .to_frame()
)

Correlation is descriptive, not evidence of causation or out-of-period reliability.

## 7. Temporal overview

Plot each split independently so gaps between collection periods are not drawn as continuous observations.

In [ ]:
# Plot periods separately so collection gaps are never shown as continuous data.
for split, frame in frames.items():
    ordered = frame.sort_values("date")
    # Use linked time axes; Light and CO2 need separate y-axes because of scale.
    fig = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        specs=[[{"secondary_y": True}], [{}], [{}]],
        subplot_titles=("Light and CO2", "Temperature and humidity", "Occupancy"),
        vertical_spacing=0.08,
    )
    # Add sensor traces in related groups, followed by the binary occupancy state.
    fig.add_trace(go.Scatter(x=ordered["date"], y=ordered["Light"], name="Light"), row=1, col=1, secondary_y=False)
    fig.add_trace(go.Scatter(x=ordered["date"], y=ordered["CO2"], name="CO2"), row=1, col=1, secondary_y=True)
    fig.add_trace(go.Scatter(x=ordered["date"], y=ordered["Temperature"], name="Temperature"), row=2, col=1)
    fig.add_trace(go.Scatter(x=ordered["date"], y=ordered["Humidity"], name="Humidity"), row=2, col=1)
    fig.add_trace(go.Scatter(x=ordered["date"], y=ordered["Occupancy"], name="Occupancy", line_shape="hv"), row=3, col=1)
    fig.update_yaxes(title_text="Light (lux)", row=1, col=1, secondary_y=False)
    fig.update_yaxes(title_text="CO2 (ppm)", row=1, col=1, secondary_y=True)
    fig.update_yaxes(title_text="Value", row=2, col=1)
    fig.update_yaxes(title_text="State", range=[-0.05, 1.05], tickvals=[0, 1], row=3, col=1)
    fig.update_layout(title=f"{split}: temporal sensor overview", height=850, hovermode="x unified")
    fig.show()

## 8. Sampling intervals and occupancy transitions

In [ ]:
# Measure timestamp regularity and count state changes within each period.
interval_summary = []
transition_summary = []

# Summarize timing gaps and occupancy transitions without crossing split boundaries.
for split, frame in frames.items():
    ordered = frame.sort_values("date")
    intervals = ordered["date"].diff().dt.total_seconds().dropna()
    transitions = ordered["Occupancy"].diff().fillna(0).ne(0)

    interval_summary.append(
        {
            "split": split,
            "median_interval_seconds": intervals.median(),
            "minimum_interval_seconds": intervals.min(),
            "maximum_interval_seconds": intervals.max(),
            "gaps_over_90_seconds": int((intervals > 90).sum()),
        }
    )
    transition_summary.append(
        {"split": split, "occupancy_transitions": int(transitions.sum())}
    )

display(pd.DataFrame(interval_summary).set_index("split"))
display(pd.DataFrame(transition_summary).set_index("split"))

In [ ]:
# Collect symmetric windows around every arrival and departure event.
transition_rows = []
window_minutes = 30

# Process each period independently so a boundary cannot create a false transition.
for split, frame in frames.items():
    ordered = frame.sort_values("date").reset_index(drop=True)
    transition_positions = np.flatnonzero(ordered["Occupancy"].diff().fillna(0).ne(0))
    for transition_number, position in enumerate(transition_positions, start=1):
        # Clip windows at file boundaries while preserving relative row offsets.
        start = max(0, position - window_minutes)
        stop = min(len(ordered), position + window_minutes + 1)
        window = ordered.iloc[start:stop].copy()
        window["minutes_from_transition"] = np.arange(start, stop) - position
        window["transition_id"] = f"{split}_{transition_number}"
        window["transition_type"] = (
            "arrival" if ordered.loc[position, "Occupancy"] == 1 else "departure"
        )
        transition_rows.append(window)

# Combine event windows only after each has its own ID and transition label.
transition_data = pd.concat(transition_rows, ignore_index=True)

# Reshape Light and CO2 together so they can share one aggregation workflow.
transition_long = transition_data.melt(
    id_vars=["minutes_from_transition", "transition_type", "transition_id"],
    value_vars=["Light", "CO2"],
    var_name="sensor",
    value_name="value",
)
# Use the median to reduce the influence of unusual individual transitions.
transition_summary_plot = (
    transition_long.groupby(["sensor", "transition_type", "minutes_from_transition"])["value"]
    .median()
    .reset_index()
)
# Plot arrivals and departures separately; the dashed line marks the event time.
fig = make_subplots(rows=1, cols=2, subplot_titles=("Light", "CO2"))
transition_colors = {"arrival": "#54A24B", "departure": "#E45756"}
for column, sensor in enumerate(["Light", "CO2"], start=1):
    for transition_type in ["arrival", "departure"]:
        subset = transition_summary_plot.loc[
            (transition_summary_plot["sensor"] == sensor)
            & (transition_summary_plot["transition_type"] == transition_type)
        ]
        fig.add_trace(
            go.Scatter(
                x=subset["minutes_from_transition"],
                y=subset["value"],
                mode="lines+markers",
                name=transition_type.title(),
                line_color=transition_colors[transition_type],
                legendgroup=transition_type,
                showlegend=column == 1,
            ),
            row=1,
            col=column,
        )
    fig.add_vline(x=0, line_dash="dash", line_color="black", row=1, col=column)
fig.update_xaxes(title_text="Minutes from transition")
fig.update_yaxes(title_text="Median reading")
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Median sensor behavior around occupancy transitions",
)
fig.show()

The transition plot treats row offsets as approximately minutes. Confirm the sampling-interval audit before interpreting it literally.

## 9. Schedule patterns

These plots test whether occupancy is strongly tied to hour of day and weekday.
Schedule features may be predictive, but they can also encode this particular
office's routine and generalize poorly when working patterns change.

In [ ]:
# Derive descriptive calendar fields from timestamps.
schedule_data = data.copy()
schedule_data["hour"] = schedule_data["date"].dt.hour
schedule_data["weekday"] = schedule_data["date"].dt.day_name()

WEEKDAY_ORDER = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]

# Calculate the occupied share for every hour within each supplied period.
hourly_occupancy = (
    schedule_data.groupby(["source_split", "hour"], observed=True)["Occupancy"]
    .agg(occupancy_rate="mean", observations="size")
    .reset_index()
)
display(hourly_occupancy.head())

fig = go.Figure()
split_colors = {"train": "#4C78A8", "test_1": "#F58518", "test_2": "#54A24B"}
for split in SOURCE_FILES:
    subset = hourly_occupancy.loc[hourly_occupancy["source_split"] == split]
    fig.add_trace(
        go.Scatter(
            x=subset["hour"],
            y=subset["occupancy_rate"],
            mode="lines+markers",
            name=split,
            line_color=split_colors[split],
            customdata=subset[["observations"]],
            hovertemplate=(
                "Hour %{x}:00<br>Occupied: %{y:.1%}"
                "<br>Observations: %{customdata[0]:,}<extra>%{fullData.name}</extra>"
            ),
        )
    )
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Occupancy rate by hour and supplied split",
    xaxis_title="Hour of day",
    yaxis_title="Occupancy rate",
    hovermode="x unified",
)
fig.update_xaxes(dtick=1)
fig.update_yaxes(tickformat=".0%", range=[0, 1])
fig.show()

# Aggregate all periods for a weekday-by-hour schedule overview.
schedule_heatmap = (
    schedule_data.groupby(["weekday", "hour"], observed=True)["Occupancy"]
    .mean()
    .unstack("hour")
    .reindex(WEEKDAY_ORDER)
)
fig = go.Figure(
    go.Heatmap(
        z=schedule_heatmap.to_numpy(),
        x=schedule_heatmap.columns,
        y=schedule_heatmap.index,
        zmin=0,
        zmax=1,
        colorscale="Blues",
        colorbar={"title": "Occupied share", "tickformat": ".0%"},
        hovertemplate="%{y}, %{x}:00<br>Occupied: %{z:.1%}<extra></extra>",
    )
)
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Occupancy schedule by weekday and hour",
    xaxis_title="Hour of day",
    yaxis_title="Weekday",
)
fig.show()


Interpret weekday patterns cautiously: the dataset covers only a short
collection period, so each weekday is represented by very few calendar dates.
The plots describe the observed office schedule rather than a universal rule.

## 10. Occupancy episode durations

An episode is a consecutive run in the same occupancy state. Episode-level
analysis reveals whether the data contains many brief transitions or long
continuous periods that could make row-level accuracy look deceptively strong.

In [ ]:
# Identify consecutive runs separately within each supplied period.
episode_frames = []
for split, frame in frames.items():
    ordered = frame.sort_values("date").copy()
    ordered["episode_id"] = ordered["Occupancy"].ne(
        ordered["Occupancy"].shift()
    ).cumsum()

    episodes = (
        ordered.groupby("episode_id", observed=True)
        .agg(
            state=("Occupancy", "first"),
            start=("date", "first"),
            end=("date", "last"),
            readings=("Occupancy", "size"),
        )
        .reset_index()
    )
    # Sampling is approximately one minute, so add one minute to include both
    # the first and final recorded minute of each episode.
    episodes["duration_minutes"] = (
        (episodes["end"] - episodes["start"]).dt.total_seconds() / 60 + 1
    )
    episodes["source_split"] = split
    episode_frames.append(episodes)

episode_data = pd.concat(episode_frames, ignore_index=True)
episode_data["state_label"] = episode_data["state"].map(
    {0: "Unoccupied", 1: "Occupied"}
)

episode_summary = (
    episode_data.groupby(["source_split", "state_label"], observed=True)[
        "duration_minutes"
    ]
    .agg(episodes="size", median_minutes="median", mean_minutes="mean", max_minutes="max")
    .round(1)
)
display(episode_summary)

fig = go.Figure()
for state in ["Unoccupied", "Occupied"]:
    for split in SOURCE_FILES:
        values = episode_data.loc[
            (episode_data["state_label"] == state)
            & (episode_data["source_split"] == split),
            "duration_minutes",
        ]
        fig.add_trace(
            go.Box(
                x=[state] * len(values),
                y=values,
                name=split,
                legendgroup=split,
                marker_color=split_colors[split],
                showlegend=state == "Unoccupied",
                boxpoints="all",
                jitter=0.25,
                pointpos=0,
                hovertemplate="%{x}<br>%{y:.1f} minutes<extra>" + split + "</extra>",
            )
        )
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Duration of consecutive occupancy episodes",
    xaxis_title="Episode state",
    yaxis_title="Duration (minutes, log scale)",
    boxmode="group",
)
fig.update_yaxes(type="log")
fig.show()


Long overnight and weekend unoccupied runs create a highly skewed
duration distribution, so the chart uses a logarithmic y-axis. Transition-level
recall should later complement row-level classification metrics.

## 11. Light-signal exceptions

Light is strongly correlated with occupancy. The exception table checks whether
that relationship is symmetric: occupied while dark versus unoccupied while
some light is detected.

In [ ]:
# Treat an exact zero reading as dark; positive readings indicate detected light.
light_audit = data.assign(
    occupied=data["Occupancy"].eq(1),
    light_detected=data["Light"].gt(0),
)

light_exception_rows = []
for split, frame in light_audit.groupby("source_split", observed=True):
    occupied = frame["occupied"]
    lit = frame["light_detected"]
    light_exception_rows.append(
        {
            "source_split": split,
            "occupied_rows": int(occupied.sum()),
            "occupied_while_dark": int((occupied & ~lit).sum()),
            "occupied_while_dark_rate": (occupied & ~lit).sum() / occupied.sum(),
            "unoccupied_rows": int((~occupied).sum()),
            "unoccupied_while_lit": int((~occupied & lit).sum()),
            "unoccupied_while_lit_rate": (~occupied & lit).sum() / (~occupied).sum(),
        }
    )

light_exceptions = pd.DataFrame(light_exception_rows).set_index("source_split")
display(light_exceptions)

# Use separate y-axis scales because occupied-while-dark is nearly zero,
# whereas unoccupied-while-lit occurs much more frequently.
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "Occupied while dark",
        "Unoccupied while lit",
    ),
)

split_order = list(SOURCE_FILES)
dark_rates = light_exceptions.loc[
    split_order, "occupied_while_dark_rate"
]
dark_counts = light_exceptions.loc[
    split_order, "occupied_while_dark"
].astype(int)
occupied_totals = light_exceptions.loc[
    split_order, "occupied_rows"
].astype(int)

lit_rates = light_exceptions.loc[
    split_order, "unoccupied_while_lit_rate"
]
lit_counts = light_exceptions.loc[
    split_order, "unoccupied_while_lit"
].astype(int)
unoccupied_totals = light_exceptions.loc[
    split_order, "unoccupied_rows"
].astype(int)

# Count labels make zero-height and extremely short bars explicit.
fig.add_trace(
    go.Bar(
        x=split_order,
        y=dark_rates,
        text=[
            f"{count:,} / {total:,}<br>({rate:.3%})"
            for count, total, rate in zip(
                dark_counts, occupied_totals, dark_rates
            )
        ],
        textposition="outside",
        cliponaxis=False,
        marker_color="#E45756",
        hovertemplate=(
            "%{x}<br>%{text}"
            "<extra>Occupied while dark</extra>"
        ),
        showlegend=False,
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Bar(
        x=split_order,
        y=lit_rates,
        text=[
            f"{count:,} / {total:,}<br>({rate:.1%})"
            for count, total, rate in zip(
                lit_counts, unoccupied_totals, lit_rates
            )
        ],
        textposition="outside",
        cliponaxis=False,
        marker_color="#72B7B2",
        hovertemplate=(
            "%{x}<br>%{text}"
            "<extra>Unoccupied while lit</extra>"
        ),
        showlegend=False,
    ),
    row=1,
    col=2,
)

fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Exceptions to the light–occupancy relationship",
    height=500,
)
fig.update_xaxes(title_text="Supplied split")
fig.update_yaxes(
    title_text="Rate among occupied rows",
    tickformat=".2%",
    range=[0, max(0.001, dark_rates.max() * 1.45)],
    row=1,
    col=1,
)
fig.update_yaxes(
    title_text="Rate among unoccupied rows",
    tickformat=".0%",
    range=[0, lit_rates.max() * 1.3],
    row=1,
    col=2,
)
fig.show()


An exact-zero threshold describes this dataset rather than a deployment
rule. Daylight, sensor noise, and different lighting policies could change the
relationship, which is why no-light models remain a required experiment.

## 12. Train/test distribution drift


In [ ]:
# Compare summary statistics across train and both held-out periods.
split_stats = data.groupby("source_split")[SENSOR_COLUMNS].agg(["mean", "std", "median"])
display(split_stats)

# Reshape the data so each sensor can occupy its own box-plot panel.
split_long = data.melt(
    id_vars="source_split",
    value_vars=SENSOR_COLUMNS,
    var_name="sensor",
    value_name="value",
)
# Draw split-specific boxes per sensor to expose location and spread shifts.
fig = make_subplots(rows=3, cols=2, subplot_titles=SENSOR_COLUMNS)
split_colors = {"train": "#4C78A8", "test_1": "#F58518", "test_2": "#54A24B"}
for index, sensor in enumerate(SENSOR_COLUMNS):
    row, col = divmod(index, 2)
    for split in SOURCE_FILES:
        values = split_long.loc[
            (split_long["sensor"] == sensor)
            & (split_long["source_split"] == split),
            "value",
        ]
        fig.add_trace(
            go.Box(
                y=values,
                name=split,
                marker_color=split_colors[split],
                boxpoints=False,
                legendgroup=split,
                showlegend=index == 0,
            ),
            row=row + 1,
            col=col + 1,
        )
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Sensor distributions across supplied periods",
    height=850,
)
fig.show()

### Quantified drift relative to training

The standardized mean difference (SMD) expresses a test–train mean difference
in pooled standard-deviation units. As a descriptive guide, absolute values
around 0.2, 0.5, and 0.8 are often called small, moderate, and large. These are
not hypothesis-test thresholds.

In [ ]:
# Quantify how far each test-period mean is from the training mean.
train_data = data.loc[data["source_split"] == "train"]
smd_rows = []
for split in ["test_1", "test_2"]:
    test_data = data.loc[data["source_split"] == split]
    for sensor in SENSOR_COLUMNS:
        pooled_std = np.sqrt(
            (train_data[sensor].var(ddof=1) + test_data[sensor].var(ddof=1)) / 2
        )
        smd_rows.append(
            {
                "source_split": split,
                "sensor": sensor,
                "standardized_mean_difference": (
                    test_data[sensor].mean() - train_data[sensor].mean()
                )
                / pooled_std,
            }
        )

smd = pd.DataFrame(smd_rows)
smd_table = smd.pivot(
    index="source_split",
    columns="sensor",
    values="standardized_mean_difference",
).reindex(columns=SENSOR_COLUMNS)
display(smd_table.round(2))

fig = go.Figure(
    go.Heatmap(
        z=smd_table.to_numpy(),
        x=smd_table.columns,
        y=smd_table.index,
        zmid=0,
        colorscale="RdBu",
        reversescale=True,
        text=np.round(smd_table.to_numpy(), 2),
        texttemplate="%{text:.2f}",
        hovertemplate="%{y} vs train<br>%{x}: %{z:.2f} SD<extra></extra>",
    )
)
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Standardized mean differences relative to training",
    xaxis_title="Sensor",
    yaxis_title="Test split",
)
fig.show()


## 13. EDA findings and modeling implications

- **Data quality:** All 20,560 rows share the expected schema. There are no
  missing cells, duplicate timestamps, or sampling gaps over 90 seconds.
- **Class balance:** Occupancy is the minority class: approximately 21% in
  training and `test_2`, but 36% in `test_1`. Report class-sensitive metrics
  separately for each held-out period.
- **Schedule dependence:** Occupancy is absent overnight and on observed
  weekends, and concentrates during working hours. Calendar features may be
  predictive but could overfit this short office schedule; compare models with
  and without them.
- **Light shortcut:** Light has the strongest target correlation (about 0.91).
  Almost no occupied rows are completely dark, but many unoccupied rows still
  have positive light readings. Light is powerful but not a complete occupancy
  rule, and its operational relationship may change.
- **CO2 response:** CO2 differs between occupancy states and changes more
  gradually around transitions than Light, making it a physically meaningful
  but delayed signal.
- **Episode structure:** Occupied episodes typically last tens of minutes,
  while unoccupied episodes include long overnight and weekend runs. Evaluate
  transition performance so long easy episodes do not dominate row metrics.
- **Redundancy:** `HumidityRatio` is derived from Temperature and Humidity.
  Retain it for the baseline, then compare performance with and without it.
- **Distribution shift:** Test periods differ from training. The largest
  standardized shifts include Temperature in `test_1` and Humidity and
  HumidityRatio in `test_2`. Keep both supplied tests separate and avoid random
  row-level splitting.